# OnionRoute Lite - Android Emulator + SSH Bridge

One-click setup: Android emulator + SSH server + cloudflared tunnel.

## Setup:
1. Click **Runtime > Run all** (GPU runtime recommended)
2. Wait ~5 min for full setup
3. Copy the SSH connection command printed at the end
4. Use `python3 tools/colab-connect.py` locally to run the test loop

## What you get:
- Headless Android 14 x86_64 emulator
- SSH access via Cloudflare tunnel (no ngrok account needed)
- Latest OnionRoute Lite APK auto-installed from CI


In [ ]:
#@title 1. Install Android SDK + Emulator + SSH { display-mode: "form" }
import os, subprocess, time
from pathlib import Path

ANDROID_SDK_ROOT = os.path.expanduser('~/android-sdk')
os.environ['ANDROID_SDK_ROOT'] = ANDROID_SDK_ROOT
os.environ['ANDROID_AVD_HOME'] = os.path.expanduser('~/.android/avd')
os.environ['PATH'] = f"{ANDROID_SDK_ROOT}/cmdline-tools/latest/bin:{ANDROID_SDK_ROOT}/emulator:{ANDROID_SDK_ROOT}/platform-tools:{os.environ['PATH']}"

def run(cmd, check=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f'FAILED: {cmd}\nstderr: {result.stderr}')
        raise RuntimeError(f'Command failed: {cmd}')
    return result

# System deps
print('[1/4] Installing system dependencies...')
run('apt-get update -qq 2>/dev/null')
run('apt-get install -y -qq qemu-kvm openssh-server curl wget unzip python3-pip > /dev/null 2>&1', check=False)
run('pip install -q requests', check=False)

# Configure SSH server
print('[1/4] Configuring SSH server...')
run('mkdir -p /run/sshd')
with open('/etc/ssh/sshd_config', 'a') as f:
    f.write('\nPort 2222\n')
    f.write('PermitRootLogin yes\n')
    f.write('PasswordAuthentication yes\n')
    f.write('PermitEmptyPasswords yes\n')
run('echo "root:onionroute" | chpasswd')
run('ssh-keygen -A')
run('/usr/sbin/sshd')
print('  SSH server running on port 2222 (root:onionroute)')

# Android SDK
print('[1/4] Installing Android SDK...')
if not Path(f'{ANDROID_SDK_ROOT}/cmdline-tools/latest/bin/sdkmanager').exists():
    run(f'mkdir -p {ANDROID_SDK_ROOT}/cmdline-tools')
    run(f'cd {ANDROID_SDK_ROOT}/cmdline-tools && curl -sL https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -o cmdline-tools.zip')
    run(f'cd {ANDROID_SDK_ROOT}/cmdline-tools && unzip -qo cmdline-tools.zip && rm -rf latest && mv cmdline-tools latest && rm cmdline-tools.zip')
    print('  SDK command-line tools installed')
else:
    print('  SDK already installed')

run('yes | sdkmanager --licenses > /dev/null 2>&1', check=False)
print('[1/4] Installing emulator + system image (~3 min)...')
run('sdkmanager --install "platform-tools" "emulator" "platforms;android-34" "system-images;android-34;default;x86_64" > /dev/null 2>&1')
print('  SDK ready')

In [ ]:
#@title 2. Create AVD + Start Emulator { display-mode: "form" }
import os, subprocess, time

AVD_NAME = 'onionroute_lite'
AVD_DIR = os.path.expanduser(f'~/.android/avd/{AVD_NAME}.avd')

run(f'echo no | avdmanager create avd -n {AVD_NAME} -k "system-images;android-34;default;x86_64" --force 2>&1', check=False)

ini_path = os.path.expanduser(f'~/.android/avd/{AVD_NAME}.ini')
with open(ini_path, 'w') as f:
    f.write(f'path={AVD_DIR}\n')
    f.write(f'path.rel=avd/{AVD_NAME}.avd\n')

config_path = f'{AVD_DIR}/config.ini'
with open(config_path, 'a') as f:
    f.write('\nhw.ramSize=2048\n')
    f.write('vm.heapSize=256\n')
    f.write('hw.camera.back=none\n')
    f.write('hw.camera.front=none\n')
    f.write('hw.audioInput=no\n')
    f.write('hw.audioOutput=no\n')

run('pkill -9 emulator 2>/dev/null; pkill -9 qemu 2>/dev/null', check=False)
time.sleep(2)

has_kvm = os.path.exists('/dev/kvm')
print(f'[2/4] KVM available: {has_kvm}')

gpu_mode = 'host' if has_kvm else 'swiftshader_indirect'
accel = '-accel on' if has_kvm else ''

emulator_cmd = f'emulator @{AVD_NAME} -no-window -no-audio -no-snapshot -no-boot-anim -no-metrics -gpu {gpu_mode} -memory 2048 {accel} 2>/tmp/emulator.log &'
print('[2/4] Starting emulator...')
os.system(emulator_cmd)

print('[2/4] Waiting for boot (1-3 min)...')
for i in range(80):
    time.sleep(3)
    result = subprocess.run('adb shell getprop sys.boot_completed 2>/dev/null', shell=True, capture_output=True, text=True)
    if result.stdout.strip() == '1':
        print(f'  Emulator booted in {i*3}s')
        break
    if i % 15 == 0 and i > 0:
        print(f'  ...still booting ({i*3}s)')
else:
    print('  WARNING: Boot timeout. Continuing anyway...')

run('adb devices')
print('  Emulator ready')

In [ ]:
#@title 3. Download & Install APK + Setup test script { display-mode: "form" }
import requests, io, zipfile, os

GITHUB_TOKEN = ''  #@param {type:"string"}
REPO = 'darkmaster0345/Onionroute-Lite'

def download_latest_apk():
    headers = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}
    runs = requests.get(f'https://api.github.com/repos/{REPO}/actions/runs?per_page=10', headers=headers).json()
    for run_info in runs.get('workflow_runs', []):
        if run_info['conclusion'] == 'success' and run_info['name'] == 'Android Build':
            run_id = run_info['id']
            print(f'  Found CI run #{run_info["run_number"]} (ID: {run_id})')
            break
    else:
        raise RuntimeError('No successful CI run found!')

    artifacts = requests.get(f'https://api.github.com/repos/{REPO}/actions/runs/{run_id}/artifacts', headers=headers).json()
    for art in artifacts.get('artifacts', []):
        if 'fdroid-debug' in art['name'] and not art['expired']:
            print(f'  Downloading {art["name"]} ({art["size_in_bytes"]/1024/1024:.1f} MB)...')
            resp = requests.get(f'https://api.github.com/repos/{REPO}/actions/artifacts/{art["id"]}/zip', headers=headers)
            with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
                for name in zf.namelist():
                    if name.endswith('.apk'):
                        zf.extract(name, '/tmp/')
                        print(f'  APK: /tmp/{name} ({os.path.getsize(f"/tmp/{name}")/1024/1024:.1f} MB)')
                        return f'/tmp/{name}'
    raise RuntimeError('No debug APK artifact found!')

print('[3/4] Downloading latest debug APK from CI...')
apk_path = download_latest_apk()

print('[3/4] Installing APK on emulator...')
result = subprocess.run(f'adb install -r {apk_path}', shell=True, capture_output=True, text=True)
print(result.stdout.strip())
if 'Success' in result.stdout:
    print('  APK installed!')
else:
    print(f'  Install failed: {result.stderr}')

# Create the remote test script
TEST_SCRIPT = '''#!/bin/bash
# OnionRoute Lite test script - runs on Colab/Kaggle
export ANDROID_SDK_ROOT="$HOME/android-sdk"
export PATH="$ANDROID_SDK_ROOT/platform-tools:$ANDROID_SDK_ROOT/emulator:$PATH"

case "$1" in
  install)
    echo "Installing APK: $2"
    adb install -r "$2"
    ;;
  launch)
    echo "Launching OnionRoute Lite..."
    adb shell am start -n org.onionroute.lite/org.onionroute.lite.MainActivity
    ;;
  logcat)
    DURATION=${2:-60}
    echo "Streaming logcat for ${DURATION}s..."
    adb logcat -c
    timeout $DURATION adb logcat -v time \\
      -s "TorService:*" "TorProcessManager:*" "TorControlConnection:*" \\
      "BandwidthTracker:*" "BridgeConfig:*" "KillSwitch:*" \\
      "AndroidRuntime:E" "System.err:W" "onionroute:*" 2>/dev/null || true
    ;;
  inspect)
    echo "=== Native libraries ==="
    adb shell find /data/app/ -path "*onionroute*/lib/*" -ls 2>/dev/null || echo "(need root)"
    echo "=== Running processes ==="
    adb shell ps -A 2>/dev/null | grep -iE "tor|onion|obfs4|snowflake" || echo "none found"
    echo "=== Tor data directory ==="
    adb shell ls -la /data/data/org.onionroute.lite/files/tor/ 2>/dev/null || echo "(need root)"
    echo "=== Torrc ==="
    adb shell cat /data/data/org.onionroute.lite/files/torrc 2>/dev/null || echo "(need root)"
    ;;
  status)
    adb shell getprop sys.boot_completed 2>/dev/null
    adb devices
    ;;
  *)
    echo "Usage: $0 {install|launch|logcat|inspect|status} [args]"
    ;;
esac
'''

with open('/tmp/orl-test.sh', 'w') as f:
    f.write(TEST_SCRIPT)
os.chmod('/tmp/orl-test.sh', 0o755)
print('  Test script ready at /tmp/orl-test.sh')

In [ ]:
#@title 4. Expose SSH via Cloudflare Tunnel { display-mode: "form" }
import subprocess, time, os, json, threading, re

print('[4/4] Installing cloudflared...')
os.system('curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')

print('[4/4] Starting SSH tunnel...')

def run_tunnel():
    os.system('cloudflared tunnel --url ssh://localhost:2222 > /tmp/cloudflared.log 2>&1')

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()
time.sleep(10)

tunnel_url = None
try:
    with open('/tmp/cloudflared.log', 'r') as f:
        for line in f:
            match = re.search(r'https?://([a-z0-9-]+\.trycloudflare\.com)', line)
            if match:
                tunnel_url = match.group(1)
except:
    pass

print()
print('=' * 60)
print('ONIONROUTE LITE - EMULATOR READY')
print('=' * 60)
if tunnel_url:
    print(f'\nSSH Tunnel Host: {tunnel_url}')
    print(f'SSH Port: 2222')
    print(f'User: root / Password: onionroute')
    print(f'\nConnect locally with:')
    print(f'  ssh -o StrictHostKeyChecking=no -p 2222 root@{tunnel_url}')
    print(f'\nOr use the test script:')
    print(f'  python3 tools/colab-connect.py --host {tunnel_url}')
else:
    print('\nChecking tunnel logs...')
    os.system('tail -20 /tmp/cloudflared.log')
    print('\nIf tunnel URL not found, check /tmp/cloudflared.log')
print('=' * 60)